# Practice Exam — 45 Questions, Weighted to the Exam Outline

## What's covered

- **45 exam-style multiple-choice questions** — the same count as the real Databricks Certified Data Engineer Associate exam (90 minutes).
- Questions are **weighted to the official section percentages**, so the mix mirrors the real exam.
- Each question gives four options A–D. The **Answer**, the **Why**, and the **Trap** (what the wrong options bank on) are hidden behind a collapsible *Show answer* — commit to a choice before you expand it.
- These are **fresh** questions — distinct from the 5 self-check questions at the end of each topic notebook (01–09).

## How to use it

1. Set a 90-minute timer. Answer all 45 *without* expanding any answer.
2. Note the ones you guessed, even if you got them right.
3. Then expand each answer and read the **Why** and the **Trap**.
4. For every miss, reread the matching section of the source notebook before retrying.

## Question mix (matches the exam weighting)

| Section | Exam weight | Questions | Source notebook |
|---|---|---|---|
| 1. Platform, Compute, Delta & UC foundations | 6% | 3 (Q1–Q3) | 01, 02 |
| 2. Data Ingestion | 21% | 9 (Q4–Q13) | 03 |
| 3. Transformations | part of 22% | 5 (Q14–Q18) | 04 |
| 3. Modeling & Pipelines | part of 22% | 5 (Q19–Q23) | 05 |
| 4. Lakeflow Jobs | 16% | 7 (Q24–Q30) | 06 |
| 5. CI/CD | 10% | 5 (Q31–Q35) | 07 |
| 6. Troubleshooting & Optimization | 10% | 4 (Q36–Q39) | 08 |
| 7. Governance & Security | 15% | 6 (Q40–Q45) | 09 |

**Scoring guide:** the real exam pass mark sits around 70% (≈32/45). If you clear 38/45 here *cold*, with the traps understood, you're ready to book it.

# 1 · Platform, Compute & Delta/UC Foundations (Section 1 — 6%)

## Q1 — Control plane vs. compute plane

**Scenario.** Which statement correctly describes where data and metadata live in the Databricks architecture?

- **A.** Notebook commands **and** customer table data both flow through the Databricks control plane.
- **B.** Classic compute runs in the customer's cloud account; the control plane holds notebooks, job/cluster config, and query metadata, while customer data is processed in the compute plane and stays in the customer's cloud storage.
- **C.** Serverless compute runs in the customer's own cloud subscription.
- **D.** Unity Catalog metadata is stored in the compute plane, next to the data.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** The **control plane** (Databricks-managed) holds the web UI, notebooks, job and cluster configuration, and query-history metadata. The **compute plane** runs the VMs that actually process data — in your cloud account for classic compute, in a Databricks-managed account for serverless. Customer data sits in your cloud storage and is processed in the compute plane; it never transits the control plane.

**Trap.** **D** — Unity Catalog metadata lives in the control-plane metastore, not next to the data. **C** — serverless VMs are Databricks-managed, not in your subscription.

</details>

## Q2 — Compute for unattended scheduled ETL

**Scenario.** A nightly PySpark ETL job runs unattended at 2 a.m. Pick the most cost-effective correct compute.

- **A.** An all-purpose cluster left running 24/7 to avoid startup time.
- **B.** A job cluster (or serverless jobs compute) that spins up for the run and terminates after.
- **C.** A serverless SQL warehouse.
- **D.** A single-user all-purpose cluster started manually each night.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** Job clusters use the cheaper **Jobs DBU** rate, start for the run, and terminate when it finishes — the default for scheduled, unattended ETL. Serverless jobs compute removes the startup wait. SQL warehouses run SQL only, not an arbitrary PySpark job.

**Trap.** **C** — a SQL warehouse cannot run a PySpark notebook/JAR task. **A** — the all-purpose DBU rate is higher and 24/7 idle time is wasted spend.

</details>

## Q3 — Time travel after VACUUM

**Scenario.** An engineer ran `VACUUM` with default retention this morning, then tried to time-travel the table to a version from 10 days ago — and the read fails because data files are missing. Why?

- **A.** Time travel only works within the current day.
- **B.** `VACUUM` permanently deleted data files older than the retention threshold (default 7 days), so versions beyond that window can no longer be reconstructed.
- **C.** `DESCRIBE HISTORY` was truncated.
- **D.** Time travel requires Photon to be enabled.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `VACUUM` permanently removes data files marked *removed* that are older than the retention window (default 7 days). Time travel reconstructs old versions from those files; once vacuumed, they're gone, so any version beyond the retention window is unreadable.

**Trap.** Assuming the transaction log alone preserves history — the log *references* files, but `VACUUM` deletes the actual Parquet, breaking old-version reads.

</details>

# 2 · Data Ingestion (Section 2 — 21%)

## Q4 — One CSV per day, pure SQL, rerunnable

**Scenario.** A vendor drops one CSV per day into S3. You want a pure-SQL load that automatically skips already-loaded files and is safe to re-run. Which?

- **A.** `spark.read.csv(...)` in a notebook with append mode.
- **B.** `COPY INTO target FROM 's3://...' FILEFORMAT = CSV`.
- **C.** Auto Loader in file-notification mode.
- **D.** A `CREATE TABLE AS SELECT` rebuilt every morning.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `COPY INTO` tracks which files it has already ingested and is **idempotent** — re-running skips loaded files. Pure SQL, ideal for low-frequency, predictable daily drops.

**Trap.** **A** — a plain read + append re-loads everything, creating duplicates. **C** — Auto Loader works but is overkill for one file/day; the exam rewards `COPY INTO` for low-frequency batch.

</details>

## Q5 — Thousands of small files every minute

**Scenario.** A card processor drops thousands of small JSON files every minute into ADLS. You need low-latency, continuous ingestion that scales to millions of files. Which?

- **A.** `COPY INTO` on a tight schedule.
- **B.** Auto Loader (`cloudFiles`).
- **C.** A nightly batch read of the whole directory.
- **D.** A JDBC pull from the processor.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** Auto Loader is built for high-volume, high-frequency incremental file ingestion — it scales to millions/billions of files, checkpoints its progress, and supports schema evolution.

**Trap.** **A** — `COPY INTO` re-lists the directory and doesn't scale to millions of files; Auto Loader's notification mode avoids repeated full listings.

</details>

## Q6 — Auto Loader slow on a huge nested prefix

**Scenario.** Auto Loader on a deeply nested prefix with billions of files is slow because each micro-batch re-lists the directory. Best fix?

- **A.** Increase `spark.sql.shuffle.partitions`.
- **B.** Switch to **file-notification mode** (a managed cloud event queue).
- **C.** Switch to `COPY INTO`.
- **D.** Add more worker nodes.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** File-notification mode subscribes to a cloud event queue (SQS+SNS, Event Grid, Pub/Sub) so new files arrive as events — no expensive directory listing. Directory-listing mode re-lists each micro-batch and degrades at huge file counts.

**Trap.** **D**/**A** — more compute or partitions doesn't help; the bottleneck is *listing*, not processing.

</details>

## Q7 — Schema evolution: auto-add new columns

**Scenario.** An upstream team occasionally adds new columns. You want Auto Loader to automatically pick up the new columns into the table and keep the stream going. Which `cloudFiles.schemaEvolutionMode`?

- **A.** `none`
- **B.** `failOnNewColumns`
- **C.** `addNewColumns` (the default)
- **D.** `rescue`

<details><summary>Show answer</summary>

**Answer: C.**

**Why.** `addNewColumns` (default) fails the stream once when it sees new columns, records them in the schema, and on restart includes them as real columns — typically paired with a job retry. `rescue` keeps extras only in `_rescued_data`; `failOnNewColumns` hard-fails; `none` ignores them.

**Trap.** **D** (`rescue`) puts new fields into `_rescued_data` only — it does **not** add real columns to the table. The requirement is to *add* columns → `addNewColumns`.

</details>

## Q8 — What `_rescued_data` captures

**Scenario.** What does Auto Loader's `_rescued_data` column hold?

- **A.** Rows that failed a `CHECK` constraint.
- **B.** Any field that didn't match the expected schema — type mismatches or extra columns — preserved as JSON so nothing is silently dropped.
- **C.** Late-arriving records.
- **D.** Duplicate rows removed during ingestion.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `_rescued_data` captures values that couldn't be parsed into the expected schema (type mismatches, unexpected columns), stored as JSON — a safety net so malformed/extra data is never silently lost.

**Trap.** Confusing it with pipeline **expectations** (data-quality quarantine) or with deduplication — different mechanisms entirely.

</details>

## Q9 — Governed CDC from Salesforce, no custom code

**Scenario.** The team wants governed, low-maintenance change-data-capture ingestion from Salesforce into Unity Catalog tables without writing custom API code. Which?

- **A.** A REST API polling loop in a notebook.
- **B.** A **Lakeflow Connect managed connector** for Salesforce.
- **C.** A JDBC read.
- **D.** `COPY INTO` against exported files.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** Lakeflow Connect **managed connectors** are Databricks-operated, governed ingestion pipelines for SaaS and database sources (Salesforce, Workday, SQL Server CDC, and more). No custom code, and data lands directly in UC.

**Trap.** **A** — a hand-rolled REST loop is the fallback only when no connector exists; the exam rewards the managed connector when one is available.

</details>

## Q10 — Parallelising a slow JDBC read

**Scenario.** A JDBC read from Postgres runs single-threaded and slow. Which options parallelise it?

- **A.** `multiLine`.
- **B.** `partitionColumn`, `lowerBound`, `upperBound`, and `numPartitions`.
- **C.** `mergeSchema`.
- **D.** `cloudFiles.format`.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** Spark's JDBC source parallelises by splitting a numeric/date `partitionColumn` range between `lowerBound` and `upperBound` into `numPartitions` slices, each its own connection. Without all four, the read is a single connection / single partition.

**Trap.** Setting `numPartitions` alone does nothing — you must also supply `partitionColumn` plus the bounds for Spark to split the range.

</details>

## Q11 — One row per array element

**Scenario.** Bronze holds JSON with an array column `items`. You need one row per array element in silver. Which function?

- **A.** `from_json`
- **B.** `explode(col("items"))`
- **C.** `to_json`
- **D.** `get_json_object`

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `explode` turns each element of an array column into its own row. `from_json` parses a JSON *string* into a struct; `get_json_object` extracts a single scalar by path; `to_json` serialises the other direction.

**Trap.** `from_json` — it parses structure but does **not** expand an array into multiple rows.

</details>

## Q12 — Query a small external table in place, always fresh

**Scenario.** Analysts need to occasionally query a small reference table in an external Postgres, always seeing the latest values, and you'd rather not copy it. Best approach?

- **A.** A nightly `COPY INTO`.
- **B.** **Lakehouse Federation** — register Postgres as a foreign catalog and query it in place.
- **C.** Auto Loader.
- **D.** A managed connector doing a full refresh every hour.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** Lakehouse Federation queries external sources in place through a **foreign catalog** — no ingestion, always-live data, still governed by Unity Catalog. Ideal for small, occasionally-queried, must-be-fresh data.

**Trap.** Choosing any ingestion path (A/C/D) when the requirement is *don't copy* + *always latest* + *infrequent* — federation is the no-copy answer.

</details>

## Q13 — Full snapshot every night, reflect current state

**Scenario.** A source emits a complete snapshot CSV of all customers every night with no change flags. Silver must reflect current state. Pattern?

- **A.** Auto Loader, append-only.
- **B.** `COPY INTO`, append.
- **C.** Load the full snapshot to bronze, then `MERGE INTO` silver to upsert by key.
- **D.** A streaming table.

<details><summary>Show answer</summary>

**Answer: C.**

**Why.** A full snapshot with no CDC markers → load it, then `MERGE INTO` to upsert by key so silver reflects current state without piling up copies.

**Trap.** **A**/**B** append-only — each night's full snapshot stacks as duplicates instead of updating existing rows.

</details>

# 3 · Data Transformations (Section 3 — part 1)

## Q14 — Deterministic dedup, keep latest

**Scenario.** Retries created duplicate `transaction_id` rows; you must keep the latest by `ingested_at`, deterministically. Which?

- **A.** `distinct()`
- **B.** `dropDuplicates(["transaction_id"])`
- **C.** `row_number()` over `partitionBy("transaction_id").orderBy(col("ingested_at").desc())`, keep `rn = 1`.
- **D.** `groupBy("transaction_id").count()`

<details><summary>Show answer</summary>

**Answer: C.**

**Why.** A window with `row_number()` and an explicit ordering deterministically selects the latest row per key. `dropDuplicates` keeps an *arbitrary* row among duplicates — not deterministic.

**Trap.** **B** — `dropDuplicates` on a subset is non-deterministic about *which* duplicate survives, so it can keep an older row.

</details>

## Q15 — 2 GB fact joined to a 5 MB dimension

**Scenario.** Joining a 2 GB fact to a 5 MB dimension; the shuffle is expensive. Best fix?

- **A.** `repartition` both sides by the join key.
- **B.** `broadcast(dim)` so the small side ships to every executor and the big side isn't shuffled.
- **C.** Increase `spark.sql.shuffle.partitions`.
- **D.** Use a cross join.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** Broadcasting the small side replicates it to all executors, so the large side isn't shuffled — the join becomes map-side (narrow). At 5 MB it's also under the default `autoBroadcastJoinThreshold` (10 MB), so AQE may broadcast it automatically.

**Trap.** **A** — repartitioning both sides still shuffles the 2 GB fact, which is exactly the cost you're trying to avoid.

</details>

## Q16 — Same columns, different order

**Scenario.** Two DataFrames have the same columns but in a different order; `union(other)` mis-aligns the data. Fix?

- **A.** `unionAll`
- **B.** `unionByName`
- **C.** A join
- **D.** `merge`

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `union`/`unionAll` align by **position**; `unionByName` aligns by **column name** (and can `allowMissingColumns`). For columns in a different order, `unionByName` is correct.

**Trap.** `union` and `unionAll` are identical in Spark (both keep duplicates) — neither fixes column ordering.

</details>

## Q17 — Post-shuffle partition count

**Scenario.** A small job emits 200 tiny files after every `groupBy` and runs slowly from overhead. Which config most directly controls the post-shuffle partition count?

- **A.** `spark.default.parallelism`
- **B.** `spark.sql.shuffle.partitions`
- **C.** `spark.executor.memory`
- **D.** `spark.sql.autoBroadcastJoinThreshold`

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `spark.sql.shuffle.partitions` (default 200) sets the number of partitions after a shuffle for DataFrame/SQL operations — hence the 200 tiny files. (AQE can coalesce these automatically when enabled.)

**Trap.** `spark.default.parallelism` — that's the RDD-API knob; SQL/DataFrame shuffles use `shuffle.partitions`.

</details>

## Q18 — Per-customer running total

**Scenario.** You need a per-customer running total of `amount` ordered by date. Which construct?

- **A.** `groupBy("customer").sum("amount")`
- **B.** `sum("amount").over(Window.partitionBy("customer").orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow))`
- **C.** `agg(sum("amount"))`
- **D.** A `pivot`

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** A window with an ordered frame (`unboundedPreceding` → `currentRow`) produces a cumulative/running total per partition **without collapsing rows**. `groupBy` collapses each group to a single total.

**Trap.** `groupBy` gives one total per customer, not a per-row running total — you lose the row-level detail.

</details>

# 4 · Modeling & Pipelines (Section 3 — part 2)

## Q19 — Expensive aggregation, hourly source, fast BI reads

**Scenario.** A BI dashboard hits a complex aggregation that's expensive to recompute on every query, but the source updates only hourly. You want fast reads and automatic refresh. Which gold object?

- **A.** A plain view.
- **B.** A materialized view.
- **C.** A streaming table.
- **D.** A temporary view.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** A **materialized view** precomputes and stores the result, refreshing automatically (incrementally where possible). BI reads hit the precomputed data — no recompute per query.

**Trap.** A plain view recomputes the expensive aggregation on *every* dashboard load; a streaming table is for append-only ingestion, not derived aggregations.

</details>

## Q20 — Append-only continuous ingestion

**Scenario.** Which object is right for append-only, continuously ingested event data that should update incrementally?

- **A.** A materialized view.
- **B.** A streaming table.
- **C.** A plain view.
- **D.** An external table.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** **Streaming tables** are designed for append-only, incremental processing of continuously-arriving data — each micro-batch appends. Materialized views are for derived/aggregated results that refresh.

**Trap.** MV vs. streaming table: a materialized view is *derived and refreshed*; a streaming table is *append ingestion*. Match the object to append-only continuous data.

</details>

## Q21 — Drop invalid rows, keep the pipeline running

**Scenario.** In a Lakeflow Spark Declarative Pipeline you want rows where `amount <= 0` removed from the output, but the pipeline should keep running and record the violation count. Which expectation?

- **A.** `expect`
- **B.** `expect or drop`
- **C.** `expect or fail`
- **D.** A `CHECK` constraint

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `EXPECT OR DROP` discards violating rows and continues, tracking the dropped count as a metric. `expect` (warn) keeps the rows; `expect or fail` aborts the update; `CHECK` constraints are on plain Delta, not pipeline expectations.

**Trap.** `expect` alone only *records* the violation — it does **not** drop the bad rows.

</details>

## Q22 — Declarative SCD Type 2 in a pipeline

**Scenario.** You need declarative change-data-capture that maintains SCD Type 2 history of a dimension inside a pipeline. Which?

- **A.** `MERGE INTO` in a notebook.
- **B.** `APPLY CHANGES INTO` with `STORED AS SCD TYPE 2`.
- **C.** `INSERT OVERWRITE`.
- **D.** `expect or fail`.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `APPLY CHANGES INTO` is the declarative CDC API in Lakeflow Spark Declarative Pipelines; it handles out-of-order data and maintains SCD Type 1 or Type 2 automatically.

**Trap.** `MERGE INTO` works but is *imperative* — you manage ordering and history yourself. The declarative-pipeline answer is `APPLY CHANGES INTO`.

</details>

## Q23 — Reject bad writes on plain Delta

**Scenario.** Outside a pipeline, every write to `silver.transactions` must reject rows where `amount` is negative, failing the write. Which?

- **A.** `expect or fail`.
- **B.** `ALTER TABLE ... ADD CONSTRAINT chk CHECK (amount >= 0)`.
- **C.** A row filter.
- **D.** A column mask.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** Delta `CHECK` constraints enforce a predicate on every write; any violating write fails. Expectations are pipeline-only; row filters and masks are governance features, not data-quality enforcement.

**Trap.** Reaching for **expectations** on a plain Delta table — those belong to declarative pipelines, not standalone tables.

</details>

# 5 · Lakeflow Jobs (Section 4 — 16%)

## Q24 — Not a task type

**Scenario.** Which of these is **not** a native Lakeflow Jobs task type?

- **A.** Notebook
- **B.** Pipeline
- **C.** SQL query / dashboard
- **D.** Cron expression

<details><summary>Show answer</summary>

**Answer: D.**

**Why.** Task types include notebook, Python script/wheel, SQL (query/dashboard/file/alert), pipeline, dbt, run-job, plus control-flow `if/else` and `for_each`. A **cron expression** is a *trigger schedule*, not a task in the graph.

**Trap.** Cron is how you *schedule* a job to run — it isn't a task node in the DAG.

</details>

## Q25 — Same task, one run per source

**Scenario.** You must run the same ingest notebook once per source system in a list, in parallel. Which control-flow construct?

- **A.** An `if/else` task.
- **B.** A `for_each` task that runs a nested task per element.
- **C.** A separate job per source.
- **D.** A `while` loop task.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `for_each` runs a nested task once per input value, optionally concurrently — exactly *same task, many inputs*.

**Trap.** Hand-building N near-identical tasks instead of using `for_each`, which is harder to maintain and doesn't scale with the list.

</details>

## Q26 — Pass a computed value between tasks

**Scenario.** An upstream task computes a row count; a downstream task needs that value to branch. How do you pass it?

- **A.** A global variable.
- **B.** `dbutils.jobs.taskValues.set(...)` upstream and `.get(...)` downstream.
- **C.** Write it to a temp file.
- **D.** A job parameter.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** **Task values** (`dbutils.jobs.taskValues`) pass small values between tasks in the same job run. The downstream task reads the value the upstream task computed.

**Trap.** **Job parameters** are fixed at submit time — they can't carry a value *computed* by an upstream task mid-run.

</details>

## Q27 — Run when a file lands, not on a clock

**Scenario.** An external system drops a file to a volume at unpredictable times; the job should run as soon as a file lands. Which trigger?

- **A.** A scheduled cron trigger.
- **B.** A **file-arrival** trigger.
- **C.** A continuous trigger.
- **D.** Manual runs.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** **File-arrival** triggers fire when new files appear in a monitored location — data-driven, not time-based. Cron would either fire too often or add latency.

**Trap.** Cron is wrong for unpredictable arrival times — you'd waste runs polling, or miss the SLA waiting for the next tick.

</details>

## Q28 — Which is a time-based trigger

**Scenario.** Which of these is a **time-based** trigger?

- **A.** A table-update trigger.
- **B.** A file-arrival trigger.
- **C.** A scheduled (cron) trigger.
- **D.** A continuous trigger.

<details><summary>Show answer</summary>

**Answer: C.**

**Why.** Scheduled/cron fires on the clock — time-based. File-arrival and table-update are **data-driven**; continuous keeps a single run always going.

**Trap.** Confusing **continuous** with scheduled — continuous is always-on, not clock-driven.

</details>

## Q29 — Retry only the flaky task

**Scenario.** One flaky task occasionally fails on a transient network error; you want only that task to retry up to 3 times without rerunning the whole DAG. Which?

- **A.** A job-level retry.
- **B.** A task-level retry policy on that task.
- **C.** A repair run.
- **D.** A `for_each` task.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** **Task-level retries** automatically re-attempt just the failing task, up to the configured count, without touching successful upstream tasks.

**Trap.** A **job-level retry** reruns from the start, redoing every successful upstream task — wasteful and slow.

</details>

## Q30 — Resume a failed multi-task job

**Scenario.** A 10-task job failed at task 7. You fixed the cause and want to resume without rerunning tasks 1–6. Which?

- **A.** Start a brand-new run.
- **B.** A **repair run** — reruns only the failed and downstream tasks, reusing successful outputs.
- **C.** Clone the job.
- **D.** A `for_each` task.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** A **repair run** restarts from the failed tasks, skipping already-successful upstream tasks — saving compute and time.

**Trap.** A brand-new run reprocesses all 10 tasks, wasting the work tasks 1–6 already completed.

</details>

# 6 · CI/CD (Section 5 — 10%)

## Q31 — The bundle manifest

**Scenario.** In an Automation Bundle, which file is the top-level manifest declaring the bundle name, targets, and resources?

- **A.** `resources.yml`
- **B.** `databricks.yml`
- **C.** `bundle.json`
- **D.** `config.toml`

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `databricks.yml` is the bundle root manifest — it defines the bundle, variables, targets (dev/test/prod), and includes the resource definition files.

**Trap.** Resources are usually split into `resources/*.yml`, but the entry point the CLI reads is `databricks.yml`.

</details>

## Q32 — Bundle command order

**Scenario.** Correct order of Databricks CLI bundle commands to push a bundle to a workspace and start it?

- **A.** `bundle run` → `bundle deploy`
- **B.** `bundle validate` → `bundle deploy` → `bundle run`
- **C.** `bundle deploy` → `bundle validate`
- **D.** `bundle destroy` → `bundle run`

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `validate` checks the config, `deploy` uploads assets and creates/updates the resources in the workspace, and `run` executes a job or pipeline.

**Trap.** Running before deploying — there's nothing in the workspace to run yet.

</details>

## Q33 — One codebase, different per environment

**Scenario.** You want one codebase that deploys a small cluster to dev and a large cluster plus a service-principal run-as to prod. Which bundle feature?

- **A.** Separate repositories per environment.
- **B.** **Targets** with environment-specific overrides (and variables).
- **C.** Notebook widgets.
- **D.** Git branches only.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** Bundle **targets** let you override resource settings per environment — compute size, run-as identity, schedules — from a single codebase.

**Trap.** Maintaining separate repos/copies defeats the single-source-of-truth that bundles exist to provide.

</details>

## Q34 — Who a production job runs as

**Scenario.** Best practice for the run-as identity of a production job?

- **A.** The deploying engineer's personal account.
- **B.** A **service principal**.
- **C.** A workspace admin account.
- **D.** A shared group account.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** Production workloads should run as a **service principal** so they don't break when an individual leaves, and so permissions are scoped, stable, and auditable.

**Trap.** Personal accounts — fragile (breaks on offboarding) and a security/audit problem.

</details>

## Q35 — What Databricks Git Folders do

**Scenario.** Which statement about Databricks Git Folders is true?

- **A.** They remove the need for an external Git provider.
- **B.** They clone a remote repo into the workspace and let you create/switch branches, commit, and push, with pull-request review done in the Git provider.
- **C.** They auto-deploy to production on every commit.
- **D.** They store data, not code.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** Git Folders integrate the workspace with a remote Git repo for branch/commit/push; PR review happens in GitHub or Azure DevOps. They don't deploy — that's bundles + the CLI in a CI pipeline.

**Trap.** Thinking Git Folders themselves perform CI/CD deployment — deployment is the job of Automation Bundles and the CLI.

</details>

# 7 · Troubleshooting & Optimization (Section 6 — 10%)

## Q36 — Reading skew in the Spark UI

**Scenario.** In the Spark UI a stage shows one task's shuffle read at 8 GB while the median is 50 MB. What does this indicate?

- **A.** A failing disk.
- **B.** **Data skew** — one partition (key) is far larger than the rest.
- **C.** Too few executors.
- **D.** Driver out-of-memory.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** A large **Max vs. Median** shuffle-read gap is the classic skew signature — one key dominates a partition, so one task does most of the work while the rest idle.

**Trap.** Adding executors doesn't help — the single skewed task still bottlenecks the stage. Fix with AQE skew-join, salting, or broadcast.

</details>

## Q37 — First skew remedy on Databricks

**Scenario.** Best first remedy for a skewed shuffle join, with AQE available?

- **A.** Manually salt every join key.
- **B.** Enable **AQE skew-join handling** (often on by default), or **broadcast** the small side if it fits.
- **C.** Increase driver memory.
- **D.** `coalesce(1)`.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** AQE's skew-join optimization splits skewed partitions automatically; if the other side is small enough, broadcasting eliminates the shuffle entirely. Salting is the manual fallback when AQE can't handle it.

**Trap.** Jumping straight to **salting** — it's more code and complexity; try AQE and broadcast first.

</details>

## Q38 — Large disk spill during aggregation

**Scenario.** The Spark UI shows large 'Spill (Disk)' during an aggregation and the stage is slow. Cause and fix?

- **A.** A network problem; add nodes.
- **B.** Not enough execution memory to hold the working set, so it spills to disk; reduce data per task, add partitions, or increase executor memory.
- **C.** Too many small input files.
- **D.** Photon is disabled.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** Spill happens when a task's working set exceeds available execution memory and overflows to (slow) disk. Remedies: more executor memory, more/smaller partitions, or less data per task.

**Trap.** Reading spill as a *disk* problem — it's a **memory-pressure** symptom, not hardware.

</details>

## Q39 — Driver OOM after collect()

**Scenario.** A job fails with a **driver** OutOfMemory right after `.collect()` on a large DataFrame. Fix?

- **A.** Increase executor memory.
- **B.** Don't pull the whole dataset to the driver — write it to a table or use `take`/`limit`; the driver isn't sized to hold all the data.
- **C.** Add more worker nodes.
- **D.** Enable AQE.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `collect()` pulls every row to the driver; on large data that OOMs the driver. The fix is to not collect everything — write the result out, or sample with `take`/`limit`.

**Trap.** Bumping **executor** memory — the failure is **driver**-side, caused by collecting all rows to one machine.

</details>

# 8 · Governance & Security (Section 7 — 15%)

## Q40 — SELECT granted but query denied

**Scenario.** A user has `SELECT` on `main.silver.customers` but queries fail with permission denied. What's missing?

- **A.** `SELECT` on the column.
- **B.** `USE CATALOG` on `main` and `USE SCHEMA` on `silver`.
- **C.** `MODIFY` on the table.
- **D.** Ownership of the table.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** To reach a table you need `USE CATALOG` on the catalog **and** `USE SCHEMA` on the schema, in addition to `SELECT` on the table. Missing a `USE` at any level blocks access.

**Trap.** Assuming table-level `SELECT` is sufficient — you must be able to *traverse* the namespace with `USE` privileges first.

</details>

## Q41 — Block one table within a granted schema

**Scenario.** A user is in a group granted `SELECT` on a schema, but must be blocked from one sensitive table. Most direct fix?

- **A.** Remove them from the group.
- **B.** `DENY SELECT` on that table to the user.
- **C.** A row filter.
- **D.** A column mask.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** `DENY` explicitly blocks a privilege and overrides `GRANT`, so you can carve out one table without touching the group grant. **DENY beats GRANT.**

**Trap.** Removing them from the group is over-broad — they'd lose access to everything else in the schema too.

</details>

## Q42 — Hide a column's value from most users

**Scenario.** Analysts may query the table but must see the card PAN masked, except the compliance group who sees it in full. Which feature?

- **A.** A row filter.
- **B.** A **column-masking** function applied to the column.
- **C.** A `CHECK` constraint.
- **D.** `DENY`.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** A **column mask** runs a function that returns masked values for non-privileged principals and the full value for allowed groups — column-level, and it preserves all rows.

**Trap.** A **row filter** removes *rows*, not values — but the requirement is to hide a *column's value* while keeping the row.

</details>

## Q43 — Each analyst sees only their region's rows

**Scenario.** Regional analysts should see only rows for their own region; global compliance sees all. Which feature?

- **A.** A column mask.
- **B.** A **row filter** function on the table, keyed off the user's group/region.
- **C.** Separate tables per region.
- **D.** `DENY`.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** A **row filter** applies a predicate per principal so each user sees only the rows they're permitted — region-scoped access from a single table.

**Trap.** A **column mask** hides *values*, it doesn't restrict *which rows* are visible — wrong tool for row-level scoping.

</details>

## Q44 — One central policy across hundreds of tables

**Scenario.** The platform team wants ONE central policy that masks every column tagged `pii = high` across hundreds of tables for everyone outside `compliance`, without editing each table. Which?

- **A.** Per-table column masks.
- **B.** **Unity Catalog ABAC** (attribute-based access control) with tags plus a single policy.
- **C.** A dynamic view per table.
- **D.** `DENY` on each table.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** UC **ABAC** lets you tag columns/objects once and write a single attribute-based policy that applies everywhere the tag appears — it scales to hundreds of tables, managed centrally.

**Trap.** Per-table masks or views give the right *behavior* but don't *scale* — ABAC is the centralized, tag-driven answer.

</details>

## Q45 — Managed vs. external table behaviour

**Scenario.** Which is true of a Unity Catalog **managed** table that is **not** true of an external table?

- **A.** It can store Delta data.
- **B.** Dropping the table also deletes the underlying data files, and UC can run predictive optimization / maintenance on it.
- **C.** It can be queried with Spark SQL.
- **D.** It can have a column mask.

<details><summary>Show answer</summary>

**Answer: B.**

**Why.** For a **managed** table, UC owns both metadata and the underlying files: `DROP` removes the data, and UC can apply predictive optimization and managed maintenance. For an **external** table, UC owns only the metadata — `DROP` leaves the files in your specified location.

**Trap.** Options A, C, D apply to *both* kinds of table — the distinguishing behaviour is file lifecycle ownership (drop-deletes-data) and managed optimization.

</details>

## Summary — how to read your score

- **38–45 cold, traps understood** → book the exam.
- **32–37** → passing range, but shaky. Reread the source notebook for every miss and retake.
- **Below 32** → work back through notebooks 01–09 before retrying; you're recognising terms but not yet picking the *purpose-built* feature under pressure.

**The five meta-patterns these 45 questions drill** — the same instincts the real exam rewards:

1. **Match the feature to the problem, not the keyword.** The exam describes a scenario; you pick the *purpose-built* tool (COPY INTO vs. Auto Loader, MV vs. streaming table, column mask vs. row filter vs. ABAC).
2. **When two answers both "work," pick the more managed / declarative / Databricks-recommended one** — managed tables, declarative pipelines, AQE, predictive optimization, service principals, ABAC.
3. **Know current product names cold** — Lakeflow Jobs, Lakeflow Spark Declarative Pipelines, Lakeflow Connect, Git Folders, Automation Bundles. Legacy names (Workflows, DLT, Repos, DABs) show up as distractors.
4. **Time-based vs. data-driven** runs through ingestion, triggers, and pipelines — cron is time, file-arrival / table-update / Auto Loader are data-driven.
5. **Cost and the right compute** — job clusters for scheduled ETL, SQL warehouses for BI, serverless for fast start, all-purpose for interactive dev only.

Good luck — when these feel obvious rather than tricky, you're ready.